# tool_eval — 도구 통합(consolidation) A/B 평가

**질문:** 각각 파라미터가 1개인 도구 A·B 가 있다. 둘을 합치니 파라미터가 3개인 도구 하나가 됐다.
이 통합은 **베스트인가, 별로인가?**

`~/Desktop/도구-eval-가이드.md` 및 Anthropic *Writing effective tools for agents* 의
**"consolidate functionality"**(자주 함께 쓰이는 도구를 합쳐 라운드트립·오류를 줄인다) 지침을,
가상의 **딥한(결정적) 도구**로 실측 비교한다. 해시는 결정적이라 에이전트가 도구 없이는 만들 수 없어
실행·채점이 명확하다.

| 설계안 | 도구 | 파라미터 |
|---|---|---|
| **분리(separate)** | `normalize_text(text)` + `hash_all(text)` | 각 1개 (에이전트가 normalize→hash 를 손으로 체이닝) |
| **통합(consolidated)** | `digest(text, normalize, algo)` | 3개 (1콜) |

같은 과제 세트를 두 설계안에 돌려 **정확도·도구호출수·토큰·도구오류**로 비교하고,
"베스트냐 별로냐"에 근거로 답한다.

## 0. 셋업

In [ ]:
import os, sys, json, time, hashlib, pathlib
from dataclasses import dataclass, field
from typing import Callable

def find_up(rel):
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        p = base / rel
        if p.exists():
            return p
    return None

try:
    from dotenv import load_dotenv
    env = find_up(".env")
    if env:
        load_dotenv(env)
except Exception:
    pass
from openai import OpenAI
MODEL = "gpt-5-nano"  # 레포 관례
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("OpenAI:", "준비됨 (실행 셀 사용 가능)" if client else "키 없음 — 실행 셀은 건너뜀")

## 딥 엔진(결정적) + 두 설계안

`normalize_text` 는 앞뒤 공백 제거 + 소문자화 + 내부 공백 정리. `_digest` 는 실제 hashlib.
- **분리안**: `normalize_text` 는 정규화만, `hash_all` 은 md5/sha1/sha256 을 전부 반환(에이전트가 골라 씀).
- **통합안**: `digest(text, normalize, algo)` 가 정규화 여부·알고리즘을 파라미터로 받아 한 번에 처리.
  `algo` 를 **enum** 으로 제약해 잘못된 값은 표현조차 안 되게 한다.

In [ ]:
def normalize_text(s):
    # 앞뒤 공백 제거 + 소문자화 + 내부 연속 공백을 하나로
    return " ".join((s or "").strip().lower().split())

def _digest(text, algo):
    algo = (algo or "").lower()
    if algo not in ("md5", "sha1", "sha256"):
        return None
    return hashlib.new(algo, text.encode("utf-8")).hexdigest()

# ── 설계안 A: 분리 (도구 2개, 각 파라미터 1개) ──
TOOLS_SEPARATE = [
    {"type": "function", "name": "normalize_text",
     "description": "입력 문자열을 정규화한다: 앞뒤 공백 제거 + 소문자화 + 내부 연속 공백을 하나로. 정규화된 문자열을 반환.",
     "parameters": {"type": "object",
        "properties": {"text": {"type": "string", "description": "정규화할 원본 문자열"}},
        "required": ["text"]}},
    {"type": "function", "name": "hash_all",
     "description": ("주어진 문자열을 그대로(정규화 없이) md5/sha1/sha256 로 해시해 세 hex 다이제스트를 JSON 으로 반환. "
                     "정규화가 필요하면 먼저 normalize_text 로 처리한 결과를 넣어라."),
     "parameters": {"type": "object",
        "properties": {"text": {"type": "string", "description": "해시할 문자열"}},
        "required": ["text"]}},
]

# ── 설계안 B: 통합 (도구 1개, 파라미터 3개) ──
TOOLS_CONSOLIDATED = [
    {"type": "function", "name": "digest",
     "description": "문자열의 해시 다이제스트를 한 번에 계산한다. normalize=true 면 정규화(공백정리+소문자) 후 해시. algo 로 알고리즘 선택.",
     "parameters": {"type": "object",
        "properties": {
            "text": {"type": "string", "description": "해시할 원본 문자열"},
            "normalize": {"type": "boolean", "description": "해시 전에 정규화할지 여부"},
            "algo": {"type": "string", "enum": ["md5", "sha1", "sha256"], "description": "해시 알고리즘"}},
        "required": ["text", "normalize", "algo"]}},
]

def sep_dispatch(name, args):
    if name == "normalize_text":
        return normalize_text(args.get("text", ""))
    if name == "hash_all":
        t = args.get("text", "")
        return json.dumps({a: _digest(t, a) for a in ("md5", "sha1", "sha256")}, ensure_ascii=False)
    return f"<error>알 수 없는 도구: {name}</error>"

def con_dispatch(name, args):
    if name == "digest":
        src = normalize_text(args.get("text", "")) if args.get("normalize") else args.get("text", "")
        d = _digest(src, args.get("algo", ""))
        if d is None:
            return f"<error>지원하지 않는 algo: {args.get('algo')}. md5|sha1|sha256 중 하나.</error>"
        return d
    return f"<error>알 수 없는 도구: {name}</error>"

print("엔진 준비. 예) sha256('hello world') =", _digest("hello world", "sha256")[:16], "…")

## 평가 과제 (Stage 1) — 두 설계안 공통

과제는 normalize 여부와 algo 를 명시한다(도구 설계 A/B 를 재는 것이지, 에이전트가 설정을 알아맞히는 걸 재는 게 아니다).
검증기는 hashlib 로 정답 hex 를 계산해 답에 그 hex 가 들어있는지만 본다(느슨).
대상 문자열은 « » 로 감싼다 — 시스템 프롬프트가 « » 를 경계로만 쓰라고 지시해, 에이전트가 따옴표를 문자열에 복사하던 교란(초기 실행에서 발견)을 없앴다.

In [ ]:
@dataclass
class Task:
    id: str
    split: str
    prompt: str
    text: str
    do_norm: bool
    algo: str

    def verify(self, answer):
        src = normalize_text(self.text) if self.do_norm else self.text
        expected = _digest(src, self.algo)
        ok = expected.lower() in (answer or "").lower()
        return ok, f"기대 {self.algo}={expected[:12]}…"

EVAL_TASKS = [
    Task("C1", "train", "문자열 «Hello World» 를 정규화(공백정리+소문자화)한 뒤 sha256 hex 다이제스트를 구하라.",
         "Hello World", True, "sha256"),
    Task("C2", "train", "«  OrderHub   API  » 를 정규화한 뒤 md5 hex 다이제스트를 구하라.",
         "  OrderHub   API  ", True, "md5"),
    Task("C3", "train", "«ImmutableExact» 를 정규화하지 말고 그대로 sha256 hex 다이제스트를 구하라.",
         "ImmutableExact", False, "sha256"),
    Task("C4", "train", "«MixEd  CaSe» 를 정규화한 뒤 sha1 hex 다이제스트를 구하라.",
         "MixEd  CaSe", True, "sha1"),
    Task("C5", "heldout", "«Raw-No-Norm-42» 를 정규화 없이 md5 hex 다이제스트를 구하라.",
         "Raw-No-Norm-42", False, "md5"),
    Task("C6", "heldout", "«  Trailing  spaces  here » 를 정규화한 뒤 sha256 hex 다이제스트를 구하라.",
         "  Trailing  spaces  here ", True, "sha256"),
]
print(f"과제 {len(EVAL_TASKS)}개 (train {sum(t.split=='train' for t in EVAL_TASKS)} / heldout {sum(t.split=='heldout' for t in EVAL_TASKS)})")

## 실행 하네스 (arm 파라미터화)

동일 루프에 `tools`·`dispatch` 만 갈아끼워 두 설계안을 같은 조건에서 비교한다.

In [ ]:
SYSTEM_PROMPT = (
    "너는 해시 계산 에이전트다. 주어진 도구만으로 정확한 hex 다이제스트를 구하라. "
    "해시 값을 절대 직접 추측하지 말고 반드시 도구로 계산하라.\n"
    "과제의 대상 문자열은 « » 로 감싸 표기한다. « 와 » 는 경계 표시일 뿐 문자열 내용이 아니다 — "
    "안쪽 내용만(앞뒤 공백 포함) 그대로 도구에 넘겨라.\n"
    "1) 도구 호출 앞에 <plan>...</plan> 으로 무엇을 할지 한 줄 적어라.\n"
    "2) 마지막에 <answer>hex</answer> 로 최종 hex 를, 이어서 <tool_feedback>...</tool_feedback> 로 "
    "도구 설계(파라미터 수·이름·enum)에서 헷갈렸거나 좋았던 점을 한 줄 남겨라.\n"
)

def run_task(client, task, tools, dispatch, arm="", model=None, max_turns=10):
    model = model or MODEL
    input_list = [{"role": "user", "content": task.prompt}]
    transcript, used = [], []
    n_calls = n_errors = in_tok = out_tok = 0
    final_text = ""
    t0 = time.time()
    for _ in range(max_turns):
        resp = client.responses.create(model=model, instructions=SYSTEM_PROMPT,
                                       input=input_list, tools=tools, parallel_tool_calls=True)
        if resp.usage:
            in_tok += resp.usage.input_tokens
            out_tok += resp.usage.output_tokens
        input_list += resp.output
        if resp.output_text.strip():
            transcript.append(("assistant", resp.output_text.strip()))
        calls = [it for it in resp.output if it.type == "function_call"]
        if not calls:
            final_text = resp.output_text
            break
        for c in calls:
            try:
                args = json.loads(c.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            try:
                result = dispatch(c.name, args)
            except Exception as e:
                result = f"<error>{type(e).__name__}: {e}</error>"
            n_calls += 1
            used.append(c.name)
            if result.startswith("<error>"):
                n_errors += 1
            transcript.append(("tool_call", f"{c.name}({json.dumps(args, ensure_ascii=False)})"))
            transcript.append(("tool_result", result[:200]))
            input_list.append({"type": "function_call_output", "call_id": c.call_id, "output": result})
    else:
        final_text = final_text or "(max_turns 도달)"
    passed, reason = task.verify(final_text)
    return {"arm": arm, "task_id": task.id, "split": task.split, "passed": passed, "reason": reason,
            "tool_calls": n_calls, "tool_errors": n_errors, "input_tokens": in_tok, "output_tokens": out_tok,
            "duration_s": round(time.time() - t0, 2), "final_text": final_text, "transcript": transcript}

## A/B 실행 & 비교 (Stage 2·3)

두 설계안을 같은 6과제에 돌려 지표를 나란히 놓는다. (12 루프 — 토큰 소모 주의)

In [ ]:
ARMS = [
    ("분리(2도구·각1파라미터)", TOOLS_SEPARATE, sep_dispatch),
    ("통합(1도구·3파라미터)", TOOLS_CONSOLIDATED, con_dispatch),
]
TRIALS = 3  # 과제당 반복 횟수 (변동 평균). 늘리면 토큰 더 씀.

def line(label, rs):
    n = len(rs)
    return (f"{label:24} 정확도 {sum(r['passed'] for r in rs)/n:4.0%} ({sum(r['passed'] for r in rs)}/{n}) · "
            f"평균 도구호출 {sum(r['tool_calls'] for r in rs)/n:.2f} · "
            f"도구오류 {sum(r['tool_errors'] for r in rs)} · "
            f"평균 토큰 {sum(r['input_tokens']+r['output_tokens'] for r in rs)/n:,.0f} · "
            f"평균 {sum(r['duration_s'] for r in rs)/n:.1f}s")

if client:
    all_results = []
    for label, tools, disp in ARMS:
        rs = [run_task(client, t, tools, disp, arm=label) for t in EVAL_TASKS for _ in range(TRIALS)]
        all_results += rs
        print(line(label, rs))
    outdir = pathlib.Path.cwd() / "results"; outdir.mkdir(exist_ok=True)
    outfile = outdir / f"consolidation-{time.strftime('%Y%m%d-%H%M%S')}.jsonl"
    outfile.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in all_results))
    print(f"\nTRIALS={TRIALS} · 저장:", outfile)
else:
    print("OPENAI_API_KEY 없음 — 이 셀 건너뜀")

## 구조 분석 (API 불필요)

지표 없이도 설계 자체로 말할 수 있는 것: 파라미터 표면과 불변식, 그리고 두 설계안이 정말 같은 결과를 내는지.

In [ ]:
def param_surface(tools):
    return {t["name"]: list(t["parameters"].get("properties", {})) for t in tools}

print("분리안 파라미터:", param_surface(TOOLS_SEPARATE))
print("통합안 파라미터:", param_surface(TOOLS_CONSOLIDATED))
print()
print("· 분리안: 도구 2개, 파라미터 총 2개. normalize→hash '순서'를 에이전트가 손으로 이어야 함(체이닝 부담).")
print("· 분리안 hash_all 은 md5/sha1/sha256 을 항상 다 반환 → 필요 없는 다이제스트도 컨텍스트에 실림(rightsizing 안 됨).")
print("· 통합안: 도구 1개, 파라미터 3개. algo 를 enum 으로 제약 → 잘못된 알고리즘은 표현 불가(무효 상태 차단).")
print("· 통합안: normalize 여부/algo 조합에 무효 조합이 없음(모든 조합이 유효) → 통합해도 혼란 없음.")

# 결정적 정합성: 두 설계안이 같은 입력에 같은 결과를 내는가
s = "  Hello   World "
sep = json.loads(sep_dispatch("hash_all", {"text": normalize_text(s)}))["sha256"]
con = con_dispatch("digest", {"text": s, "normalize": True, "algo": "sha256"})
print("\n두 설계안 동일 결과:", sep == con, "(", sep[:16], "… )")

## 결론 — 1→3 파라미터 통합, 베스트야 별로야?

**"자주 함께 쓰이는(체이닝되는) 두 도구를 합치는 것"은 대체로 베스트다.** Anthropic 의 *consolidate functionality* 권장과 이 eval 이 함께 가리키는 방향:

- ✅ **호출 수·라운드트립 감소** — 분리안은 normalize→hash 를 에이전트가 손으로 이어야 하고, 체이닝 지점마다 "정규화를 깜빡"·"중간 출력 오독" 오류 여지가 생긴다. 통합안은 1콜.
- ✅ **토큰 절약** — 분리안 `hash_all` 은 3종 다이제스트를 다 뱉어 컨텍스트를 더 먹는다(rightsizing 안 됨). 통합안은 필요한 것만.
- ✅ **불변식으로 무효 상태 차단** — `algo` 를 enum 으로 두면 잘못된 알고리즘을 아예 표현 못 한다("pass the intern test").

**하지만 '별로'가 되는 조건도 분명하다 — 3개가 됐다는 사실 자체가 문제는 아니고, 그 3개가 어떤지가 문제다:**
- ❌ 파라미터들이 **무효 조합**을 만들 수 있으면(서로 배타적인 플래그 등) 오히려 혼란 → enum·required 로 막아야.
- ❌ 3번째 파라미터가 **거의 항상 같은 값**이면 표면만 늘린 bloat → 기본값 있는 단순 도구가 낫다.
- ❌ A·B 가 **독립적으로도 자주 유용**하면 합치는 순간 유연성을 잃는다.

### 답
> **둘이 늘 함께 쓰이고(체이닝), 3번째 파라미터가 실제로 의미 있는 선택지이며, 무효 조합이 없다면 — 통합이 베스트다.**
> 이 케이스(normalize→hash 파이프라인 + `normalize` 플래그 + `algo` enum)는 세 조건을 다 만족하므로 **통합이 낫다.**
> 위 A/B 실행 지표(정확도·호출수·토큰·오류)가 그 근거다. 반대로 세 조건 중 하나라도 깨지면 "별로"로 뒤집힌다 — 그래서 **"합쳤더니 3개"라는 사실만으로는 좋다/나쁘다를 못 정하고, 반드시 eval 로 확인**해야 한다.